# Deliverable 4 — Entrenamiento QLoRA Multi-Agente (`Enrutador`, `Recuperador`, `Redactor`) + Publicación HF
Notebook oficial de entrenamiento para **Deliverable 4 (`3-LoRA`)**:
- Base única: `Qwen/Qwen3-4B` cuantizado 4-bit NF4 (`bf16` en A100/L4, `fp16` en T4).
- **3 adaptadores independientes entrenados por rol** mediante `scripts/entrenar_agente.py` (`transformers.Trainer` + `peft`, `completion_only_loss = True`, `LoRA r=16, alpha=32, dropout=0.05`, `0` ejemplos truncados):
  - `enrutador`: `210` train / `36` val (`max_seq_length=2048`, `eval_steps=10`).
  - `recuperador`: `230` train / `39` val (`max_seq_length=2048`, `eval_steps=15`).
  - `redactor`: `180` train / `30` val (`max_seq_length=4096`, `eval_steps=15`).
- **Paso 5 (`publicar_adaptadores_hf.py`):** calcula el SHA-256 de `adapter_model.safetensors` antes de subir a Hugging Face, verifica que el hash remoto coincida tras la subida y actualiza `training_metrics.json` y `entrenamiento/manifest_adaptadores_hf.json` (`redactor` ya publicado en `Pablox0333/qwen3-4b-redactor-udec-d4` @ `739b031`).

In [ ]:
import subprocess
subprocess.run(["nvidia-smi", "-L"], check=False)
%pip -q install -U transformers==5.23.0 peft==0.21.3 bitsandbytes==0.50.2 accelerate==1.15.0 huggingface-hub==1.33.0 matplotlib

import torch

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    bf16_ok = torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported()
    print(f"GPU: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB | bf16={bf16_ok}")

## Paso 2 — Obtener código de `deliverable-4` y verificar datasets antifuga (725 pares)

In [ ]:
import hashlib
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1.git"
RAMA = "deliverable-4"
WORKDIR = Path("/content/Proyecto_GenIA")

if WORKDIR.exists():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", RAMA], check=True)
    subprocess.run(["git", "-C", str(WORKDIR), "checkout", RAMA], check=True)
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only", "origin", RAMA], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA, REPO_URL, str(WORKDIR)], check=True)

D4_DIR = WORKDIR / "Deliverable 4"
SCRIPT_ENTRENAR = D4_DIR / "scripts" / "entrenar_agente.py"
SCRIPT_PUBLICAR = D4_DIR / "scripts" / "publicar_adaptadores_hf.py"
assert SCRIPT_ENTRENAR.is_file() and SCRIPT_PUBLICAR.is_file()

print("Commit clonado:", subprocess.check_output(["git", "-C", str(WORKDIR), "rev-parse", "--short", "HEAD"], text=True).strip())
for rol in ("enrutador", "recuperador", "redactor"):
    tr = D4_DIR / "datos" / f"{rol}_train.jsonl"
    va = D4_DIR / "datos" / f"{rol}_val.jsonl"
    ntr = sum(1 for l in tr.read_text(encoding="utf-8").splitlines() if l.strip())
    nva = sum(1 for l in va.read_text(encoding="utf-8").splitlines() if l.strip())
    print(f"[{rol:<11}] {ntr} train / {nva} val | sha_tr={hashlib.sha256(tr.read_bytes()).hexdigest()[:12]} sha_va={hashlib.sha256(va.read_bytes()).hexdigest()[:12]}")

## Paso 3 — Entrenar uno o los 3 agentes con `scripts/entrenar_agente.py`
- Si solo quieres publicar los adaptadores ya evaluados en A100, **salta directamente al Paso 5**.
- Si quieres reentrenar desde cero, edita `AGENTES_A_ENTRENAR` y ejecuta esta celda.

In [ ]:
from pathlib import Path
import subprocess

AGENTES_A_ENTRENAR = ["enrutador", "recuperador", "redactor"]
USAR_DRIVE = True
OUT_ROOT = Path("/content/drive/MyDrive/UdeC_GenIA_D4_3LoRA/entrenamiento") if USAR_DRIVE else Path("/content/d4_3lora_entrenamiento")

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Carpeta destino de entrenamiento:", OUT_ROOT)

for agente in AGENTES_A_ENTRENAR:
    out_agente = OUT_ROOT / agente
    cmd = [
        "python3",
        str(SCRIPT_ENTRENAR),
        "--agente", agente,
        "--output-dir", str(out_agente),
    ]
    print(f"\n=== Entrenando {agente} -> {out_agente} ===")
    subprocess.run(cmd, check=True)

## Paso 4 — Inspeccionar curvas de pérdida y `smoke_test_val.json` por rol

In [ ]:
import json
from IPython.display import Image, display
from pathlib import Path

BASE_INSPECCION = OUT_ROOT if "OUT_ROOT" in globals() else Path("/content/drive/MyDrive/UdeC_GenIA_Exp_MultiLoRA/entrenamiento")
for agente in ("enrutador", "recuperador", "redactor"):
    out_agente = BASE_INSPECCION / agente
    mpath = out_agente / "training_metrics.json"
    if not mpath.is_file():
        continue
    m = json.loads(mpath.read_text(encoding="utf-8"))
    print(f"\n[{agente}] {m['hardware']['gpu']} | resultados={json.dumps(m['resultados'], ensure_ascii=False)}")
    png = out_agente / "loss_curve.png"
    if png.is_file():
        display(Image(filename=str(png)))

## Paso 5 — Publicar `Enrutador` y `Recuperador` en Hugging Face (verificando SHA-256 antes y después)
- Pon `PUBLICAR_CORRIDA_ORIGINAL_A100 = True` para publicar **exactamente los pesos evaluados en A100** (`/content/drive/MyDrive/UdeC_GenIA_Exp_MultiLoRA/entrenamiento`).
- Pon `PUBLICAR_CORRIDA_ORIGINAL_A100 = False` para publicar los recién entrenados en `OUT_ROOT`.
- El `Redactor` ya vive públicamente en `Pablox0333/qwen3-4b-redactor-udec-d4` (`739b031`).

In [ ]:
import json
import os
import subprocess
from pathlib import Path
from google.colab import drive
from huggingface_hub import notebook_login

drive.mount("/content/drive")
HF_USERNAME = "Pablox0333"
PUBLICAR_CORRIDA_ORIGINAL_A100 = True
CARPETA_PESOS_HISTORICA = Path("/content/drive/MyDrive/UdeC_GenIA_Exp_MultiLoRA/entrenamiento")

if PUBLICAR_CORRIDA_ORIGINAL_A100:
    DRIVE_PUBLICAR = CARPETA_PESOS_HISTORICA
else:
    assert "OUT_ROOT" in globals(), "Ejecuta primero el Paso 3 o activa PUBLICAR_CORRIDA_ORIGINAL_A100=True"
    DRIVE_PUBLICAR = OUT_ROOT

assert DRIVE_PUBLICAR.is_dir(), f"No existe carpeta de adaptadores: {DRIVE_PUBLICAR}"
print("Publicando adaptadores desde:", DRIVE_PUBLICAR)

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception as exc:
        print("Aviso leyendo Colab Secret HF_TOKEN:", repr(exc))
        token = None
if not token:
    print("HF_TOKEN no encontrado o sin permiso en Secrets; iniciando login interactivo...")
    notebook_login()
else:
    os.environ["HF_TOKEN"] = token

cmd_pub = [
    "python3",
    str(SCRIPT_PUBLICAR),
    "--hf-username", HF_USERNAME,
    "--roles", "enrutador,recuperador",
    "--drive-dir", str(DRIVE_PUBLICAR),
]
print("Ejecutando:", " ".join(cmd_pub))
subprocess.run(cmd_pub, check=True)

manifest_path = D4_DIR / "entrenamiento" / "manifest_adaptadores_hf.json"
print("\n=== Manifiesto HF actualizado (copia al repo) ===")
print(manifest_path.read_text(encoding="utf-8"))

## Paso 6 — Empaquetar manifiesto, métricas con SHA-256 y curvas para sincronizar el repo local

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

bundle_zip = Path("/content/D4_manifest_y_metricas_hf.zip")
archivos = [
    D4_DIR / "entrenamiento" / "manifest_adaptadores_hf.json",
    D4_DIR / "entrenamiento" / "enrutador" / "training_metrics.json",
    D4_DIR / "entrenamiento" / "enrutador" / "adapter" / "README.md",
    D4_DIR / "entrenamiento" / "recuperador" / "training_metrics.json",
    D4_DIR / "entrenamiento" / "recuperador" / "adapter" / "README.md",
]
with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in archivos:
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(D4_DIR)))

print("Paquete generado:", bundle_zip)
files.download(str(bundle_zip))